In [ ]:
import pandas as pd
import db_dtypes
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, brier_score_loss
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

In [9]:
def build_preprocessor(df_raw):

    analysis_cols = [
    'patient_age', 'gender', 'length_of_stay', 'main_code', 'num_diagnoses',
    'num_chronic_conditions', 'num_procedures', 'has_diabetes', 'has_cancer',
    'has_hiv', 'has_hf', 'has_alz', 'has_ckd', 'had_surgery', 'admission_cost',
    'total_procedure_costs', 'total_medication_costs', 'total_stay_cost', 
    'admissions_365d', 'tot_length_of_stay_365d', 'avg_cost_of_prev_stays',
    'is_planned', 'following_unplanned_admission_flag', 'readmit_30d', 'readmit_90d'
    ]

    df_numeric = df_raw[analysis_cols]

    df_numeric['avg_cost_of_prev_stays'] = df_numeric['avg_cost_of_prev_stays'].fillna(0)
    df_numeric['following_unplanned_admission_flag'] = df_numeric['following_unplanned_admission_flag'].fillna(0)

    df_numeric = pd.get_dummies(df_numeric)
    df_numeric = df_numeric.drop(columns = 'gender_F')

    mask = df_numeric['following_unplanned_admission_flag'] == 0
    df_numeric.loc[mask, ['readmit_30d', 'readmit_90d']] = 0
    mask = df_numeric['readmit_90d'] == 0
    df_numeric.loc[mask, 'following_unplanned_admission_flag'] = 0

    df_results = df_numeric[['readmit_30d', 'readmit_90d']]
    df_numeric['log_stay_cost'] = np.log(df_numeric['total_stay_cost'])
    df_numeric['log_prev_avg_stay_cost'] = np.log1p(df_numeric['avg_cost_of_prev_stays'])
    df_numeric['log_total_procedure_costs'] = np.log1p(df_numeric['total_procedure_costs'])
    df_numeric['log_total_medication_costs'] = np.log1p(df_numeric['total_medication_costs'])

    df_numeric = df_numeric.drop(columns = ['total_stay_cost', 'avg_cost_of_prev_stays', 'total_procedure_costs', 'total_medication_costs'
,'readmit_30d', 'readmit_90d', 'following_unplanned_admission_flag', 'main_code'])

    return df_numeric, df_results

In [10]:
def make_train_test_split(X, y):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 0)

    return X_train, X_test, y_train, y_test

In [11]:
df_raw = pd.read_csv("D:\Python Projects\Hospital readmission risk\index_stay.csv", sep = ',')
df_numeric, df_results = build_preprocessor(df_raw)

C:\Users\4infi\AppData\Local\Temp\ipykernel_29632\3613884750.py:1: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("D:\Python Projects\Hospital readmission risk\index_stay.csv", sep = ',')
C:\Users\4infi\AppData\Local\Temp\ipykernel_29632\2235083803.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_numeric['avg_cost_of_prev_stays'] = df_numeric['avg_cost_of_prev_stays'].fillna(0)
C:\Users\4infi\AppData\Local\Temp\ipykernel_29632\2235083803.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-doc

In [12]:
X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_90d'])

In [13]:
scaler = StandardScaler()

data = scaler.fit_transform(X_train)

In [14]:
rf = RandomForestClassifier()

rf_param_dist = {
    "n_estimators": [100, 200, 300, 500, 800, 1000],
    "max_depth": [None, 4, 6, 8, 10],
    "min_samples_split": [2, 10, 20, 50, 100],
    "min_samples_leaf": [1, 5, 10, 20, 50],
    "max_features": ["sqrt", 0.3, 0.5],
    "class_weight": ["balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="average_precision",   # or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

rf_search.fit(data, y_train)

best_rf_params_90 = rf_search.best_params_
best_rf_score_90 = rf_search.best_score_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


KeyboardInterrupt: 

In [10]:
best_rf_params_90

{'n_estimators': 300,
 'min_samples_split': 2,
 'min_samples_leaf': 5,
 'max_features': 0.5,
 'max_depth': None,
 'class_weight': 'balanced_subsample'}

In [11]:
best_rf_score_90

np.float64(0.8300736808674732)

In [15]:
rf = RandomForestClassifier()

In [16]:
rf_param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 1],
    "min_samples_split": [2, 2, 5],
    "min_samples_leaf": [2, 5, 10],
    "max_features": [0.3, 0.5, 0.6, 0.7],
    "class_weight": ["balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="average_precision",   # or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

rf_search.fit(data, y_train)

rf_search.best_score_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


np.float64(0.8259554162189672)

In [17]:
rf_search.best_params_

{'n_estimators': 500,
 'min_samples_split': 2,
 'min_samples_leaf': 2,
 'max_features': 0.6,
 'max_depth': None,
 'class_weight': 'balanced_subsample'}

After this rerun everything

In [19]:
X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_30d'])
data = scaler.fit_transform(X_train)

rf_param_dist = {
    "n_estimators": [300, 500, 800],
    "max_depth": [None, 4, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [2, 5, 10],
    "max_features": ["sqrt", 0.3, 0.5, 0.6, 0.7],
    "class_weight": ["balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="average_precision",   # or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

rf_search.fit(data, y_train)

best_rf_params_30 = {**rf_search.best_params_, 'random_state': 42}
best_rf_score_30 = rf_search.best_score_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


In [20]:
best_rf_score_30

np.float64(0.6259391816844352)

In [21]:
best_rf_params_30

{'n_estimators': 500,
 'min_samples_split': 10,
 'min_samples_leaf': 5,
 'max_features': 0.5,
 'max_depth': None,
 'class_weight': 'balanced_subsample',
 'random_state': 42}

In [15]:
rf_param_dist = {
    "n_estimators": [400, 500, 800, 1000],
    "max_depth": [None, 4],
    "min_samples_split": [5, 50],
    "min_samples_leaf": [5, 50],
    "max_features": [0.5, 0.6, 0.7, 0.8],
    "class_weight": ["balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="average_precision",   # or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

rf_search.fit(data, y_train)

best_rf_params_30 = {**rf_search.best_params_, 'random_state': 42}
best_rf_score_30 = rf_search.best_score_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


KeyboardInterrupt: 

In [17]:
X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_90d'])
data = scaler.fit_transform(X_train)

In [28]:
lgbm = LGBMClassifier(objective="binary", force_col_wise = True, random_state=42)

lgbm_param_dist = {
    "n_estimators": [100, 200, 400, 600, 1000],
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127],
    "max_depth": [-1, 6, 8, 10],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_samples": [10, 20, 40],
    "reg_lambda": [0.0, 1.0, 5.0],
    "reg_alpha": [0.0, 0.5, 1.0],
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_param_dist,
    n_iter=40,
    scoring="average_precision",   # again: or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

lgbm_search.fit(data, y_train)  # same 30d training set or a stratified subset

best_lgbm_params_90 = lgbm_search.best_params_
best_lgbm_score_90 = lgbm_search.best_score_

Fitting 3 folds for each of 40 candidates, totalling 120 fits
[LightGBM] [Info] Number of positive: 3610, number of negative: 73042
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 76652, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047096 -> initscore=-3.007327
[LightGBM] [Info] Start training from score -3.007327


In [29]:
best_lgbm_params_90

{'subsample': 0.8,
 'reg_lambda': 0.0,
 'reg_alpha': 1.0,
 'num_leaves': 63,
 'n_estimators': 100,
 'min_child_samples': 20,
 'max_depth': -1,
 'learning_rate': 0.05,
 'colsample_bytree': 0.7}

In [30]:
best_lgbm_score_90

np.float64(0.6423838902137919)

In [31]:
lgbm_param_dist = {
    "n_estimators": [50, 100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [45, 64, 90],
    "max_depth": [-1, 1],
    "subsample": [ 0.7, 0.8, 0.9],
    "colsample_bytree": [0.6, 0.7, 0.8],
    "min_child_samples": [10, 20, 30],
    "reg_lambda": [0.0, 0.5],
    "reg_alpha": [0.5, 1, 2, 5],
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_param_dist,
    n_iter=40,
    scoring="average_precision",   # again: or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

lgbm_search.fit(data, y_train)

lgbm_search.best_score_

Fitting 3 folds for each of 40 candidates, totalling 120 fits
[LightGBM] [Info] Number of positive: 3610, number of negative: 73042
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 76652, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047096 -> initscore=-3.007327
[LightGBM] [Info] Start training from score -3.007327


np.float64(0.641179861306672)

In [32]:
lgbm_search.best_params_

{'subsample': 0.9,
 'reg_lambda': 0.0,
 'reg_alpha': 2,
 'num_leaves': 90,
 'n_estimators': 200,
 'min_child_samples': 10,
 'max_depth': -1,
 'learning_rate': 0.03,
 'colsample_bytree': 0.7}

In [36]:
X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_30d'])
data = scaler.fit_transform(X_train)

lgbm_param_dist = {
    "n_estimators": [400, 500, 600, 700],
    "learning_rate": [0.01, 0.03, 0.05],
    "num_leaves": [63, 90, 128],
    "max_depth": [-1, 1],
    "subsample": [0.5, 0.6, 0.7],
    "colsample_bytree": [0.4, 0.5, 0.6],
    "min_child_samples": [1, 2, 5, 10],
    "reg_lambda": [0.0, 0.5, 1],
    "reg_alpha": [0.0, 0.1],
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_param_dist,
    n_iter=40,
    scoring="average_precision",   # again: or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

lgbm_search.fit(data, y_train)  # same 30d training set or a stratified subset

best_lgbm_params_30 = lgbm_search.best_params_
best_lgbm_score_30 = lgbm_search.best_score_

Fitting 3 folds for each of 40 candidates, totalling 120 fits
[LightGBM] [Info] Number of positive: 3610, number of negative: 73042
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 76652, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047096 -> initscore=-3.007327
[LightGBM] [Info] Start training from score -3.007327


In [37]:
best_lgbm_params_30


{'subsample': 0.6,
 'reg_lambda': 1,
 'reg_alpha': 0.1,
 'num_leaves': 63,
 'n_estimators': 700,
 'min_child_samples': 2,
 'max_depth': -1,
 'learning_rate': 0.01,
 'colsample_bytree': 0.5}

In [38]:
best_lgbm_score_30

np.float64(0.6418225091099283)

In [42]:
lgbm_params = {
    'subsample': 0.6,
 'reg_lambda': 1,
 'reg_alpha': 0.1,
 'num_leaves': 63,
 'n_estimators': 700,
 'min_child_samples': 2,
 'max_depth': -1,
 'learning_rate': 0.01,
 'colsample_bytree': 0.5,
    "class_weight": "balanced",
    "random_state": 42,
}

lgb_general = LGBMClassifier(objective="binary", force_col_wise = True, **lgbm_params)

lgb_general.fit(data, y_train)

y_pred = lgb_general.predict_proba(data)[:, 1]
average_precision_score(y_train, y_pred)

[LightGBM] [Info] Number of positive: 3610, number of negative: 73042
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 76652, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.7048616961626053

In [46]:
roc_auc_score(y_train, y_pred)

0.9826494523964164

In [45]:
lgbm_params = {
    'subsample': 0.6,
 'reg_lambda': 0.5,
 'reg_alpha': 0.0,
 'num_leaves': 64,
 'n_estimators': 400,
 'min_child_samples': 5,
 'max_depth': -1,
 'learning_rate': 0.03,
 'colsample_bytree': 0.5
}

lgb_general_new = LGBMClassifier(objective="binary", force_col_wise = True, **lgbm_params)

lgb_general_new.fit(data, y_train)

y_pred = lgb_general_new.predict_proba(data)[:, 1]

average_precision_score(y_train, y_pred)

[LightGBM] [Info] Number of positive: 3610, number of negative: 73042
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 76652, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047096 -> initscore=-3.007327
[LightGBM] [Info] Start training from score -3.007327


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.8369223857642509